In [ ]:
import numpy as np
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity

In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


### Basic ReasonIR-style RAG implementation.
This code shows the core technical idea of the ReasonIR paper.

ReasonIR idea:
A normal retriever should find documents that help answer
a difficult question, even when the same words are not used.

Pipeline:
Question
- Query rewrite
- Query embedding
- Document embedding
- Cosine similarity
- Top documents
- RAG-style answer

In [ ]:

documents = [
    {
        "id": 1,
        "title": "Circadian Rhythm",
        'text': (
            "Circadian rhythm is the natural 24-hour biological"
            " clock of the human body.It controls sleep, alertness,"
            "body temperature, hormone release, and energy levels."
            "People who work night shifts often struggle to sleep"
            "during the day because their body clock is misaligned"
            "with sunlight and normal sleep timing"
        ),
    },
    {
        'id' : 2,
        "title": "Corporate Restructuring",
        "text": (
            "Corporate restructuring is a business process where a"
            "company changes its internal structure, departments,"
            "roles, ro financial model to improve efficiency or "
            "reduce costs."

        ),
    },
    {
        "id":3,
        "title": "wrongful termination",
        "text": (
            "Wrongful termination happens when an employee is fired"
            "illegally, such as after reporting misconduct, harassment,"
            "fruad, or unsafe practices. Retaliation after protected"
            "reporting can create legal liability for the employer."
        ),
    },
    {

        "id": 4,
        "title": "Machine Learning Overfitting",
        "text": (
            "Overfitting happens when a machine learning model "
            "memorizes training data instead of learning general "
            "patterns. Such a model performs well on training data "
            "but poorly on new data."
        ),
    },
    {
        "id": 5,
        "title": "Retrieval Augmented Generation",
        "text": (
            "Retrieval Augmented Generation, or RAG, improves "
            "language model answers by first retrieving relevant "
            "documents from a knowledge base. These documents are "
            "then passed as context to the LLM."
        ),

    },
]

#### 2. Sample ReasonIR-style training data

This is not full training
It only shows how ReasonIR-style data is structured

Each training example has:
1. Query <br>
    A reasoning-heavy question.

2. Positive document    <br>
    The correct useful document.

3. Hard_negative_document <br>
    A confusing document that looks related, <br>
    but does not actually help.

Why we use this: <br>
The ReasonIR Paper trains the retriever with contrastive learning.

The retriever learns:
- query should be close to the positive document
- query should be far from the hard negative document


In [ ]:
training_examples = [
  {
   "query": (
    "A person works at night and feels tired during work"
    "but cannot sleep properly during the day"
    "what biological concept explains this?"
   ),
   "Positive_document":(
    "Circadian rhythm controls sleep, alertness,"
    "and body clock timing "
   ),
   "hard_negative_document": (
    "corporate restructing changes employee roles"
    "and business departments"
   ),
  },
  {
  "query": (
    "An employee reports unethical behavior and is fired"
    "soon after. What legal concept may apply?"
  ),
    "positive_document": (
            "Wrongful termination and retaliation laws protect "
            "employees who report misconduct."
        ),
        "hard_negative_document": (
            "Corporate restructuring is used to improve "
            "company efficiency."
        ),
    },
    {
        "query": (
            "Why does a model perform well on training data "
            "but badly on unseen data?"
        ),
        "positive_document": (
            "Overfitting means the model memorized training data "
            "instead of learning general patterns."
        ),
        "hard_negative_document": (
            "RAG retrieves documents before generating an answer."
        ),
    },

 ]

### 3. Query rewritting

Query rewriting expandsa short user question into a richer query.

Why we use this:

Reasoning questions may not contain the exact words
used inside the useful document

Example:<br>
User asks:<br>
"Why do night-shift workers sleep badly?"

Useful document may talk about:
- circadian rhythm
- body clock
- sleep cycle
- sunlight

Rewriting helps the retriever search for hidden concepts

In the real ReasonIR Paper, an LLM rewrites the query.
Here we use a simple rule-based rewrite.


In [ ]:
def rewrite_query(query: str) -> str:
    """
    Rewrite the original question into a richer retrievel query

    Args:
        query:
            Original user question.
    Returns:
        A more detailed query for better retrieval
        """
    rewritten = (
        "Reasoning query: \n"
        f"{query}\n\n"
        "Retrieve documents that explain:\n"
        "- the hidden concept behind the question\n"
        "- the background knowledge needed \n"
        "- The cause-effect relationship\n"
        "- the theory that helps answer it\n"
    )
    return rewritten

#### 4. Basic ReasonIR-style retriever

 This class does the main retrieval work.

 It handles:
 - loading the embedding model
 - converting documents into vectors
- converting user query into a vector
- comparing query and document vectors
- returning top matching documents
#
Why we use embeddings: <br>
Computers do not understand raw text meaning directly.<br>
Embeddings convert text into numbers.<br>
Similar meanings should have similar vectors

In [ ]:
class ReasonIRBasicRetriever:
    def __init__(
            self,
            documents,
            model_name="sentence-transformers/all-MiniLM-L6-v2",
    ):
        """Initialize the retriever.
         Args:
            documents:
                List of documents to search from.

            model_name:
                Embedding model name.

        Why this model:
            all-MiniLM-L6-v2 is small and fast.
            It is good for a basic local demo.
            """

        self.documents =documents

        #load embedding model
        # this converts text into vector embeddings.
        self.model = SentenceTransformer(model_name)

        # Combine title and text

        # why:
        # Title gives strong topic signal
        # Body gives detailed knowledge
        self.document_texts = []

        for doc in documents:
            full_text = doc['title'] + "\n" + doc['text']
            self.document_texts.append(full_text)

        print ('Encoding documents...')


        # Convert documents into embeddings
        # Why:
        # Document embeddings can be created once
        # and reused for every user query.
        #
        # normalize_embeddings=True makes cosine
        # similarity more stable.
        self.document_embeddings = self.model.encode(
            self.document_texts,
            normalize_embeddings =True
        )

    def retrieve(
            self,
            query,
            top_k=3,
            use_query_rewrite=True,
    ):
        """
        Retrieve top-k useful documents.

        Args:
            query:
                User question.

            top_k:
                Number of documents to return.

            use_query_rewrite:
                Whether to rewrite the query.

        Returns:
            List of retrieved documents with scores.

        """
        if use_query_rewrite:
            query = rewrite_query(query)

        # Encode the query
        query_embedding = self.model.encode(query, normalize_embeddings=True)

        # Calculate cosine similarity
        similarities = cosine_similarity(
            [query_embedding],
            self.document_embeddings
        )[0]

        # Sort documents by similarity and get top-k
        ranked_docs = []
        for i, doc in enumerate(self.documents):
            ranked_docs.append({
                'id': doc['id'],
                'title': doc['title'],
                'text': doc['text'],
                'similarity_score': similarities[i]
            })

        # Sort by similarity score in descending order
        ranked_docs.sort(key=lambda x: x['similarity_score'], reverse=True)

        return ranked_docs[:top_k]

#### 5. Simple RAG answer generator

RAG means Retrieval -Augmented Generation.

In a real RAG system.
1. Retrieve useful documents
2. Send question + documents to an LLM
3. LLM generates final answer

Here we are not calling an LLM.<br>
We simply show the top retrieved document as the answer.

Why:<br>
This keeps the implementation simple and easy to understand.

In [ ]:
def generate_answer(question, retrieved_docs):
    """
    Generate a basic answer using retrieved documents.
    Args:
        question:
            Original user question.

        retrieved_docs:
            Documents returned by the retriever.

    Returns:
        A simple RAG-style answer string.
        """
    # Combine retrieved documents into one context block.
    #
    # In production, this context is passed to an LLM.
    context_parts = []

    # Handle case where retrieved_docs might be None or empty
    if not retrieved_docs:
        context = "No relevant documents found."
        top_doc_title = "N/A"
        top_doc_text = "No explanation available as no documents were retrieved."
    else:
        for doc in retrieved_docs:
            part=(
                f"Document:{doc['title']}\n"
                f"{doc['text']}"
            )
            context_parts.append(part) # Fix: use 'part' instead of 'context_part'
        context = "\n\n".join(context_parts)

        #The first document has highest score.
        top_doc = retrieved_docs[0]
        top_doc_title = top_doc['title']
        top_doc_text = top_doc['text']


    answer=(
        "\nQuestion:\n"
        f"{question}\n\n"
        "Retrieved context:\n"
        f"{context} \n\n"
        "Basic RAG Answer:\n"
        f"The most relevant concept is: {top_doc_title}\n\n" # Fix: use 'top_doc_title' instead of 'top'
        "Explanation:\n"
        f"{top_doc_text}\n"
    )
    return answer

#### 6. Test the implementation
This block runs only when this file executed directly.

why:
This allows the file to be used in two ways:

1. Run directly:
     Python reasonir_basic.py

2. Import into another app:
    from reasonir_basic import ReasonIRBasicRetriever



In [ ]:
if __name__ == "__main__":
    #create retriever object
    #This loads the model and encodes all documents
    retriever = ReasonIRBasicRetriever(documents)

    #Sample reasoning-style qeustions
    #these questions do not always use exact document titles
    #The retriever should still find right concept

    sample_queries = [
        (
            "Why do night-shift workers struggle"
            "to sleep during the day?"
        ),
        (
            "A company fired an employee after they "
            "reported misconduct. What issue is this?"
        ),
        (
            "Why does my ML model work well on training "
            "data but fail on new data?"
        ),
        "How does RAG improve LLM answers?",
    ]
    print("\n ReasonIR-style Basic Retriever Ready\n")

    for query in sample_queries:
        print("="* 70)
        print("User Query :")
        print(query)

        # Retrieve top 2 documents
        retrieved_docs = retriever.retrieve(
            query=query,
            top_k=2,
            use_query_rewrite = True,
        )

        #Generate basic RAG-style answer
        answer = generate_answer(
            question=query,
            retrieved_docs = retrieved_docs,
        )

        print('\n Generated Answer:')
        print(answer)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Encoding documents...

 ReasonIR-style Basic Retriever Ready

User Query :
Why do night-shift workers struggleto sleep during the day?

 Generated Answer:

Question:
Why do night-shift workers struggleto sleep during the day?

Retrieved context:
Document:Circadian Rhythm
Circadian rhythm is the natural 24-hour biological clock of the human body.It controls sleep, alertness,body temperature, hormone release, and energy levels.People who work night shifts often struggle to sleepduring the day because their body clock is misalignedwith sunlight and normal sleep timing

Document:Retrieval Augmented Generation
Retrieval Augmented Generation, or RAG, improves language model answers by first retrieving relevant documents from a knowledge base. These documents are then passed as context to the LLM. 

Basic RAG Answer:
The most relevant concept is: Circadian Rhythm

Explanation:
Circadian rhythm is the natural 24-hour biological clock of the human body.It controls sleep, alertness,body temperat